In [135]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


## Read/data

In [136]:
DATA_PATH = "/home/denklewer/Data/schreiber_full/"
# read total data

total_df =  pd.read_csv("{}schreiber_total_data_full.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(total_df['allele']))
per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt = pd.read_csv("{}schreiber_data_full_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    df_wt = pd.read_csv("{}schreiber_data_full_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";") 
    
    per_allele_mt_df[allele_name] = df_mt
    per_allele_wt_df[allele_name] = df_wt
    
    assert 'peptide' in df_mt.columns
    assert 'length' in df_mt.columns
    assert RESULT_COLUMNS['id'] in df_mt.columns
    assert 'peptide' in df_wt.columns
    assert 'length' in df_wt.columns
    assert RESULT_COLUMNS['id'] in df_wt.columns
ALLELES

['H-2-Kd', 'H-2-IEd', 'H-2-Dd', 'H-2-IAd', 'H-2-Ld']

In [137]:
per_method_per_allele_wt_predictions = dict()
per_method_per_allele_mt_predictions = dict()

## 0.1 Define TARGET_ALLELES (alleles than should be passed to algorithm)

In [138]:
TARGET_ALLELES = dict()

## HARDCODE alelle name which will be passed to algorithm
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    else:
        target_allele_name = allele_name
    TARGET_ALLELES[allele_name] = (target_allele_name)


TARGET_ALLELES

{'H-2-Kd': 'H-2-Kd',
 'H-2-IEd': 'H-2-IEd',
 'H-2-Dd': 'H-2-Dd',
 'H-2-IAd': 'H-2-IAd',
 'H-2-Ld': 'H-2-Ld'}

# 1. NetMHC-3.4

## 1.0 Define methods

In [139]:
from mhctools import NetMHC3
METHOD = 'netMHC-3.4'

def splitByLengthAndFilter(fasta_dict, min_length = 8):
    clean_fasta_dict = {k: v for k, v in fasta_dict.items() if not pd.isnull(v)}  #delete nan elements
    unique_lengths = list(set(map(len, list(clean_fasta_dict.values()))))
    unique_lengths = list(filter(lambda x: x>=min_length, unique_lengths ))
    per_length_dict_of_dicts = {target_length :{k: v for k, v in clean_fasta_dict.items() if len(v) == target_length} for target_length in unique_lengths}
    return per_length_dict_of_dicts

def getPredictor(allele_name):
    alleles = [allele_name]
    predictor = NetMHC3(
        alleles=alleles,
        program_name=METHOD)
    return predictor

def getSupportedAlleles():
    supported_alleles = NetMHC3._determine_supported_alleles(METHOD,"-A")
    return supported_alleles
    
    
def getSamplePredictionDf():
    protein_sequences = {
      "1L2Y": "NLYIQWLKD",
      "1L3Y": "ECDTINCER"
    }
    predictor = getPredictor("A*02:01")
    df = getPrediction(predictor=predictor, fasta_dict=protein_sequences, length=9)
    return df

def getPrediction(predictor, fasta_dict, length):
    predictions = predictor.predict_subsequences(fasta_dict,peptide_lengths=[length]).to_dataframe()
    return predictions
    
    
    
def fillDummyPrediction(fasta_dict, allele_name, length):
    sample_df = pd.DataFrame()
    sample_df['source_sequence_name'] = fasta_dict.keys()
    sample_df['offset'] = 0
    sample_df['peptide'] = fasta_dict.values()
    sample_df['allele'] = allele_name
    sample_df['score'] = np.nan
    sample_df['affinity'] = np.nan
    sample_df['percentile_rank'] = np.nan
    sample_df['prediction_method_name'] = np.nan
    sample_df['prediction_method_name'] = METHOD
    sample_df['length'] = length

    return sample_df
    
    
def predictSequences(dict_of_dicts, allele_name):
    supported_alleles = getSupportedAlleles()
    norm_allele_name = normalize_allele_name(TARGET_ALLELES[allele_name])# normalize allele name here 
    print(f"Predict binders for  dataset {allele_name} using alleles parameter {norm_allele_name}")
    if norm_allele_name in supported_alleles:
        predictor = getPredictor(norm_allele_name)
        binding_predictions = [getPrediction(predictor, fasta_dict, length)
                               for length, fasta_dict in dict_of_dicts.items()]
    else:
        print("!!!Unsupported allele, create dummy prediction!!!")
        binding_predictions = [fillDummyPrediction(fasta_dict,allele_name, length) for length, fasta_dict in dict_of_dicts.items()]
                    
    lengths = [len(prediction) for prediction in binding_predictions] 
    print(f"total processed peptides after filtering -  {sum(lengths)}")
    return binding_predictions

def predict_with_netmhc(per_allele_df):
    # make fasta-style dicts
   
    per_allele_fasta = {
        allele_name : {
            ID : PEPTIDE 
            for ID, PEPTIDE in zip(per_allele_df[allele_name][RESULT_COLUMNS['id']].values, per_allele_df[allele_name]['peptide'].values) 
        } 
        for allele_name in ALLELES 
    }
        # split by length
    per_allele_per_length_dict = dict() 
    for allele_name in ALLELES:
        fasta_dict = per_allele_fasta[allele_name]
        per_length_dict_of_dicts = splitByLengthAndFilter(fasta_dict)
        #print(f"allele {allele_name} fasta splited into {len(per_length_dict_of_dicts)} dicts")
        per_allele_per_length_dict[allele_name] = per_length_dict_of_dicts

    # predict
    per_allele_predictions = dict()
    for allele_name, per_length_dict in per_allele_per_length_dict.items():
        print(f"initial length {len(per_allele_df[allele_name])}")

        predictions = predictSequences(per_length_dict, allele_name)
        per_allele_predictions[allele_name] = predictions
    # join lengths together
    per_allele_total_predictions = dict()
    for allele_name, list_of_df in per_allele_predictions.items():
        df_result = pd.concat(list_of_df, ignore_index=True)
        per_allele_total_predictions[allele_name] = df_result
    return per_allele_total_predictions
    

## 1.1. Check supported alleles

In [140]:
supported_alleles = getSupportedAlleles()

for allele_name in ALLELES:
    norm_allele_name = normalize_allele_name(TARGET_ALLELES[allele_name])
    if norm_allele_name in supported_alleles:
        print(f"Allele {allele_name}: OK")
    else:
        print(f"Allele {allele_name}: is not supported!")
        


Allele H-2-Kd: OK
Allele H-2-IEd: is not supported!
Allele H-2-Dd: OK
Allele H-2-IAd: is not supported!
Allele H-2-Ld: OK


## 1.2. Predict

In [141]:
getSamplePredictionDf()

,source_sequence_name,offset,peptide,allele,score,affinity,percentile_rank,prediction_method_name,length
0,1L3Y,0,ECDTINCER,HLA-A*02:01,0.053,28329.0,None,netMHC-3.4,9
1,1L2Y,0,NLYIQWLKD,HLA-A*02:01,0.056,27238.0,None,netMHC-3.4,9


In [142]:
%%time
print(f"\t Mutated:")
per_allele_mt_predictions = predict_with_netmhc(per_allele_mt_df)
per_method_per_allele_mt_predictions[METHOD] = per_allele_mt_predictions

print(f"\t Wild:")
per_allele_wt_predictions = predict_with_netmhc(per_allele_wt_df)
per_method_per_allele_wt_predictions[METHOD] = per_allele_wt_predictions


	 Mutated:
initial length 8738
Predict binders for  dataset H-2-Kd using alleles parameter H-2-Kd
total processed peptides after filtering -  8738
initial length 52091
Predict binders for  dataset H-2-IEd using alleles parameter H-2-IEd
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  52091
initial length 8738
Predict binders for  dataset H-2-Dd using alleles parameter H-2-Dd
total processed peptides after filtering -  8738
initial length 52091
Predict binders for  dataset H-2-IAd using alleles parameter H-2-IAd
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  52091
initial length 8738
Predict binders for  dataset H-2-Ld using alleles parameter H-2-Ld
total processed peptides after filtering -  8738
	 Wild:
initial length 8738
Predict binders for  dataset H-2-Kd using alleles parameter H-2-Kd
total processed peptides after filtering -  8738
initial length 52091
Predict binders for  dataset H-2-IEd

# 2. NetMHC-4.0

In [143]:
from mhctools import NetMHC4
METHOD = 'netMHC-4.0'

In [144]:
def getPredictor(allele_name):
    alleles = [allele_name]
    predictor = NetMHC4(
        alleles=alleles,
        program_name=METHOD)
    return predictor
def getSupportedAlleles():
    supported_alleles = NetMHC4._determine_supported_alleles(METHOD,"-listMHC")
    return supported_alleles

In [145]:
getSamplePredictionDf()

,source_sequence_name,offset,peptide,allele,score,affinity,percentile_rank,prediction_method_name,length
0,1L3Y,0,ECDTINCER,HLA-A*02:01,0.026,37794.71,75.0,netMHC-4.0,9
1,1L2Y,0,NLYIQWLKD,HLA-A*02:01,0.092,18478.21,25.0,netMHC-4.0,9


## 2.1. Check alleles

In [146]:
supported_alleles = getSupportedAlleles()

for allele_name in ALLELES:
    norm_allele_name = normalize_allele_name(TARGET_ALLELES[allele_name])
    if norm_allele_name in supported_alleles:
        print(f"Allele {allele_name}: OK")
    else:
        print(f"Allele {allele_name}: is not supported!")

Allele H-2-Kd: OK
Allele H-2-IEd: is not supported!
Allele H-2-Dd: OK
Allele H-2-IAd: is not supported!
Allele H-2-Ld: OK


## 2.2. Predict

In [147]:
%%time
print(f"\t Mutated:")
per_allele_mt_predictions = predict_with_netmhc(per_allele_mt_df)
per_method_per_allele_mt_predictions[METHOD] = per_allele_mt_predictions

print(f"\t Wild:")
per_allele_wt_predictions = predict_with_netmhc(per_allele_wt_df)
per_method_per_allele_wt_predictions[METHOD] = per_allele_wt_predictions


	 Mutated:
initial length 8738
Predict binders for  dataset H-2-Kd using alleles parameter H-2-Kd
total processed peptides after filtering -  8738
initial length 52091
Predict binders for  dataset H-2-IEd using alleles parameter H-2-IEd
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  52091
initial length 8738
Predict binders for  dataset H-2-Dd using alleles parameter H-2-Dd
total processed peptides after filtering -  8738
initial length 52091
Predict binders for  dataset H-2-IAd using alleles parameter H-2-IAd
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  52091
initial length 8738
Predict binders for  dataset H-2-Ld using alleles parameter H-2-Ld
total processed peptides after filtering -  8738
	 Wild:
initial length 8738
Predict binders for  dataset H-2-Kd using alleles parameter H-2-Kd
total processed peptides after filtering -  8738
initial length 52091
Predict binders for  dataset H-2-IEd

# 3. NetMHCPan-4.1

In [148]:
from mhctools import NetMHCpan41

In [149]:
METHOD = 'netMHCpan-4.1'

In [150]:
def getPredictor(alleles):
    """
    mode - "elution_score" or "binding_affinity"
    """
    predictor_ba = NetMHCpan41(
        alleles=alleles,
        program_name=METHOD,
        mode="binding_affinity"
    )
    
    predictor_el = NetMHCpan41(
        alleles=alleles,
        program_name=METHOD,
        mode="elution_score"
    )
    return predictor_ba, predictor_el

def getSupportedAlleles():
    supported_alleles = NetMHCpan41._determine_supported_alleles(METHOD,"-listMHC")
    return supported_alleles

def reorderColumns(df):
    df = df[['source_sequence_name', 'offset', 'peptide', 'length', 'allele', 'score_ba', 'score_el', 'affinity',
                      'percentile_rank_ba', 'percentile_rank_el', 'prediction_method_name' ]]
    return df

def getPrediction(predictor, fasta_dict, length):
    predictor_ba, predictor_el = predictor
    df_ba = predictor_ba.predict_subsequences(fasta_dict, peptide_lengths=[length]).to_dataframe()
    df_el = predictor_el.predict_subsequences(fasta_dict, peptide_lengths=[length]).to_dataframe()
    
    df_ba = df_ba.rename(columns={"percentile_rank": "percentile_rank_ba"})
    df_ba = df_ba.rename(columns={"score": "score_ba"})
    df_ba["percentile_rank_el"] = df_el["percentile_rank"]
    df_ba["score_el"] = df_el["score"]
    
    predictions = reorderColumns(df_ba)
    return predictions

def fillDummyPrediction(fasta_dict, allele_name, length):
    sample_df = pd.DataFrame()
    sample_df['source_sequence_name'] = fasta_dict.keys()
    sample_df['offset'] = 0
    sample_df['peptide'] = fasta_dict.values()
    sample_df['allele'] = allele_name
    sample_df['score_ba'] = np.nan
    sample_df['score_el'] = np.nan
    sample_df['affinity'] = np.nan
    sample_df['percentile_rank_el'] = np.nan
    sample_df['percentile_rank_ba'] = np.nan
    sample_df['prediction_method_name'] = METHOD
    sample_df['length'] = length
    return reorderColumns(sample_df)
    

In [151]:
getSamplePredictionDf()

,source_sequence_name,offset,peptide,length,allele,score_ba,score_el,affinity,percentile_rank_ba,percentile_rank_el,prediction_method_name
0,1L3Y,0,ECDTINCER,9,HLA-A*02:01,0.017415,0.000061,41413.08,77.963,48.2,netMHCpan-4.1
1,1L2Y,0,NLYIQWLKD,9,HLA-A*02:01,0.074361,0.000089,22364.03,29.463,43.0,netMHCpan-4.1


## 2.1. Check alleles

In [152]:
supported_alleles = getSupportedAlleles()

for allele_name in ALLELES:
    norm_allele_name = normalize_allele_name(TARGET_ALLELES[allele_name])
    if norm_allele_name in supported_alleles:
        print(f"Allele {allele_name}: OK")
    else:
        print(f"Allele {allele_name}: is not supported!")

Allele H-2-Kd: OK
Allele H-2-IEd: is not supported!
Allele H-2-Dd: OK
Allele H-2-IAd: is not supported!
Allele H-2-Ld: OK


## 2.2. Predict

In [153]:
%%time
print(f"\t Mutated:")
per_allele_mt_predictions = predict_with_netmhc(per_allele_mt_df)
per_method_per_allele_mt_predictions[METHOD] = per_allele_mt_predictions

print(f"\t Wild:")
per_allele_wt_predictions = predict_with_netmhc(per_allele_wt_df)
per_method_per_allele_wt_predictions[METHOD] = per_allele_wt_predictions

	 Mutated:
initial length 8738
Predict binders for  dataset H-2-Kd using alleles parameter H-2-Kd
total processed peptides after filtering -  8738
initial length 52091
Predict binders for  dataset H-2-IEd using alleles parameter H-2-IEd
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  52091
initial length 8738
Predict binders for  dataset H-2-Dd using alleles parameter H-2-Dd
total processed peptides after filtering -  8738
initial length 52091
Predict binders for  dataset H-2-IAd using alleles parameter H-2-IAd
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  52091
initial length 8738
Predict binders for  dataset H-2-Ld using alleles parameter H-2-Ld
total processed peptides after filtering -  8738
	 Wild:
initial length 8738
Predict binders for  dataset H-2-Kd using alleles parameter H-2-Kd
total processed peptides after filtering -  8738
initial length 52091
Predict binders for  dataset H-2-IEd

# 4. NetMHCIIPan-4.0

In [154]:
from mhctools import NetMHCIIpan4
METHOD = 'netMHCIIpan-4.0'

In [155]:
def getPredictor(alleles):
    """
    mode - "elution_score" or "binding_affinity"
    """
    predictor_ba = NetMHCIIpan4(
        alleles=alleles,
        program_name=METHOD,
        mode="binding_affinity"
    )
    
    predictor_el = NetMHCIIpan4(
        alleles=alleles,
        program_name=METHOD,
        mode="elution_score"
    )
    return predictor_ba, predictor_el

def getSamplePredictionDf():
    protein_sequences = {
      "1L2Y": "NLYIQWLKD",
      "1L3Y": "ECDTINCER"
    }
    predictor = getPredictor("DRB1_0101")
    df = getPrediction(predictor=predictor, fasta_dict=protein_sequences, length=9)
    return df

def getSupportedAlleles():
    supported_alleles = NetMHCIIpan4._determine_supported_alleles(METHOD,"-list")
    return supported_alleles


    

In [156]:
getSamplePredictionDf()

,source_sequence_name,offset,peptide,length,allele,score_ba,score_el,affinity,percentile_rank_ba,percentile_rank_el,prediction_method_name
0,1L3Y,0,ECDTINCER,9,HLA-DRA1*01:01-DRB1*01:01,0.023452,0.000001,38794.46,95.0,95.0,netMHCIIpan-4.0
1,1L2Y,0,NLYIQWLKD,9,HLA-DRA1*01:01-DRB1*01:01,0.088652,0.000050,19160.06,95.0,95.0,netMHCIIpan-4.0


## 4.1. Check alleles

In [157]:
supported_alleles = getSupportedAlleles()

for allele_name in ALLELES:
    norm_allele_name = normalize_allele_name(TARGET_ALLELES[allele_name])
    if norm_allele_name in supported_alleles:
        print(f"Allele {allele_name}: OK")
    else:
        print(f"Allele {allele_name}: is not supported!")

Allele H-2-Kd: is not supported!
Allele H-2-IEd: OK
Allele H-2-Dd: is not supported!
Allele H-2-IAd: OK
Allele H-2-Ld: is not supported!


## 4.2. Predict

In [158]:
%%time
print(f"\t Mutated:")
per_allele_mt_predictions = predict_with_netmhc(per_allele_mt_df)
per_method_per_allele_mt_predictions[METHOD] = per_allele_mt_predictions

print(f"\t Wild:")
per_allele_wt_predictions = predict_with_netmhc(per_allele_wt_df)
per_method_per_allele_wt_predictions[METHOD] = per_allele_wt_predictions


	 Mutated:
initial length 8738
Predict binders for  dataset H-2-Kd using alleles parameter H-2-Kd
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  8738
initial length 52091
Predict binders for  dataset H-2-IEd using alleles parameter H-2-IEd
total processed peptides after filtering -  52091
initial length 8738
Predict binders for  dataset H-2-Dd using alleles parameter H-2-Dd
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  8738
initial length 52091
Predict binders for  dataset H-2-IAd using alleles parameter H-2-IAd
total processed peptides after filtering -  52091
initial length 8738
Predict binders for  dataset H-2-Ld using alleles parameter H-2-Ld
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  8738
	 Wild:
initial length 8738
Predict binders for  dataset H-2-Kd using alleles parameter H-2-Kd
!!!Unsupported allele, create dummy prediction!!!
total 

# 5. Join predictions

In [159]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'netMHCIIpan-4.0':['source_sequence_name', 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el']
}



def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df


def getPredictionResult(total_results, allele_name, method):
    df = total_results[method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df
    

# def getPredictionResult(total_results, region, allele_name, method):
#     df = total_results[region][method][allele_name][per_method_columns[method]].copy()
#     df = rename_columns(df, method)
#     return df


In [160]:
def joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions):
    per_allele_result = dict()
    methods = list(per_method_per_allele_mt_predictions.keys())
    for allele_name in ALLELES:
        print(f"allele {allele_name}")
        df_source_wt = per_allele_wt_df[allele_name].copy()
        df_source_mt = per_allele_mt_df[allele_name].copy()

        for method in methods:
            print(f"method {method}")
            df_method_wt = getPredictionResult(per_method_per_allele_wt_predictions, allele_name, method)
            df_method_mt = getPredictionResult(per_method_per_allele_mt_predictions, allele_name, method)
            df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
            df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
        leave_columns = [ 'source_sequence_name', 'allele', 'length']
        df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
        df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
        # rename id back 


        df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
        per_allele_result[allele_name] = df_result
    return per_allele_result
                
                

              

In [161]:
per_allele_result = joinPredictionsToSource(per_method_per_allele_mt_predictions, per_method_per_allele_wt_predictions)

allele H-2-Kd
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele H-2-IEd
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele H-2-Dd
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele H-2-IAd
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0
allele H-2-Ld
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
method netMHCIIpan-4.0


In [162]:
per_allele_result[ALLELES[0]][['allele', 'length','peptide_mt','peptide_wt', 'netMHC-3.4_score_mt', 'netMHC-3.4_score_wt', 'netMHCpan-4.1_score_ba_wt']]

,allele,length,peptide_mt,peptide_wt,netMHC-3.4_score_mt,netMHC-3.4_score_wt,netMHCpan-4.1_score_ba_wt
0,H-2-Kd,9,KYLTQRENL,KYLTQREKL,0.709,0.682,0.517459
1,H-2-Kd,9,VYQVERERL,VYQVELERL,0.423,0.355,0.370518
2,H-2-Kd,9,RFAPQAELL,RFPPQAELL,0.193,0.098,0.271672
3,H-2-Kd,9,APSNSGAGL,APSNSGADL,0.091,0.086,0.094032
4,H-2-Kd,9,SFVAVVITI,SFVAAVITI,0.524,0.488,0.514683
...,...,...,...,...,...,...,...
8733,H-2-Kd,8,PTTVCQPN,ATTVCQPN,0.031,0.035,0.024494
8734,H-2-Kd,8,PRLHHSSR,PLLHHSSR,0.032,0.034,0.019557
8735,H-2-Kd,11,PPRLHHSSRSC,PPLLHHSSRSC,0.030,0.034,0.018418
8736,H-2-Kd,8,PNNISAIS,PNNISGIS,0.030,0.031,0.012530


In [163]:
per_allele_result[allele_name].columns

Index(['source_sequence_name', 'allele', 'peptide_mt', 'length',
       'netMHC-3.4_score_mt', 'netMHC-3.4_affinity_mt',
       'netMHC-3.4_percentile_rank_mt', 'netMHC-4.0_score_mt',
       'netMHC-4.0_affinity_mt', 'netMHC-4.0_percentile_rank_mt',
       'netMHCpan-4.1_score_ba_mt', 'netMHCpan-4.1_score_el_mt',
       'netMHCpan-4.1_affinity_mt', 'netMHCpan-4.1_percentile_rank_ba_mt',
       'netMHCpan-4.1_percentile_rank_el_mt', 'netMHCIIpan-4.0_score_ba_mt',
       'netMHCIIpan-4.0_score_el_mt', 'netMHCIIpan-4.0_affinity_mt',
       'netMHCIIpan-4.0_percentile_rank_ba_mt',
       'netMHCIIpan-4.0_percentile_rank_el_mt', 'peptide_wt',
       'netMHC-3.4_score_wt', 'netMHC-3.4_affinity_wt',
       'netMHC-3.4_percentile_rank_wt', 'netMHC-4.0_score_wt',
       'netMHC-4.0_affinity_wt', 'netMHC-4.0_percentile_rank_wt',
       'netMHCpan-4.1_score_ba_wt', 'netMHCpan-4.1_score_el_wt',
       'netMHCpan-4.1_affinity_wt', 'netMHCpan-4.1_percentile_rank_ba_wt',
       'netMHCpan-4.1_percent

In [164]:
per_allele_result[allele_name][['peptide_mt', 'peptide_wt', 'netMHCpan-4.1_score_ba_mt', 'netMHCpan-4.1_score_el_mt', 'netMHCpan-4.1_percentile_rank_ba_mt','netMHCpan-4.1_percentile_rank_el_mt', 'netMHCpan-4.1_percentile_rank_ba_wt', 'netMHCpan-4.1_percentile_rank_el_wt']]

,peptide_mt,peptide_wt,netMHCpan-4.1_score_ba_mt,netMHCpan-4.1_score_el_mt,netMHCpan-4.1_percentile_rank_ba_mt,netMHCpan-4.1_percentile_rank_el_mt,netMHCpan-4.1_percentile_rank_ba_wt,netMHCpan-4.1_percentile_rank_el_wt
0,ALSGVSVYF,ALSGVPVYF,0.121462,0.032015,3.314,2.889,2.797,1.779
1,QPAAAPAAAL,QPPAAPAAAL,0.192317,0.119270,1.167,0.955,1.431,0.913
2,LSGVSVYF,LSGVPVYF,0.064475,0.004609,12.314,12.012,11.791,7.420
3,QVLGGVNL,QVLGRVNL,0.048159,0.006607,19.212,9.658,17.105,8.633
4,AAAPAAAL,PAAPAAAL,0.071622,0.022105,10.178,3.873,10.269,6.740
...,...,...,...,...,...,...,...,...
8733,REPAPEWPSIN,REPAPEWASIN,0.068037,0.006626,11.182,9.640,10.236,11.382
8734,DKGKIEDTGGS,EKGKIEDTGGS,0.003305,0.000001,97.312,98.333,97.278,98.333
8735,NDFSVLPLPN,NEFSVLPLPN,0.026085,0.000032,38.916,66.667,43.625,77.500
8736,EPAPEWPSINL,EPAPEWASINL,0.130738,0.119928,2.814,0.950,2.862,1.509


# 5. For now just write to files

In [165]:
METHOD = 'NetMHC'

TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [166]:
# for decoy_key in DECOY_PREFIXES:

for allele_name in ALLELES:
    total_df = per_allele_result[allele_name]
    allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}{}_predictions_allele_{}.csv".format(TARGET_PATH_TO_PREDICTIONS, METHOD,allele_name),sep=";", index=False)